In [1]:
!pip install datasets scikit-learn tqdm regex scipy

In [2]:
import re
import math
import numpy
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    name="indiccorp_v2",
    split="hin_Deva",
    streaming=True  # Optional: avoid full download
)

In [5]:
data_list=[]

for i,example in enumerate(dataset):
  data_list.append(example["text"])
  if i>3000:
    break

train_data=data_list[:2000]
valid_data=data_list[2000:2500]
test_data=data_list[2500:]

print(len(train_data),len(valid_data),len(test_data))

2000 500 502


In [15]:

import re

def sentence_tokenizer(text):
    # Split on ., !, ? for sentences
    sentences = re.split(r'(?<=[.!?]) +', text)
    return [s.strip() for s in sentences if s.strip()]

def word_tokenizer(sentence):
    # Matches:
    # - URLs
    # - Emails
    # - Dates (dd/mm/yyyy)
    # - Numbers (with decimals)
    # - Hindi words (Devanagari letters)
    # - English words
    # - Punctuation
    tokens = re.findall(
        r'https?://\S+|'           # URLs
        r'\S+@\S+|'                 # Emails
        r'\d{2}/\d{2}/\d{4}|'       # Dates
        r'\d+\.\d+|\d+|'            # Numbers
        r'[\u0900-\u097F]+|'        # Hindi words
        r'[a-zA-Z]+|'                # English words
        r'[^\w\s]',                  # Punctuation
        sentence,
        flags=re.UNICODE
    )
    return tokens


In [22]:
# ============================================
# Step 4: Tokenize data
# ============================================
def tokenize_dataset(data):
    tokenized = []
    for text in data:
        sentences = sentence_tokenizer(text)
        for sent in sentences:
            tokens = word_tokenizer(sent)
            tokenized.append(tokens)
    return tokenized

train_tokens = tokenize_dataset(train_data)
valid_tokens = tokenize_dataset(valid_data)
test_tokens  = tokenize_dataset(test_data)

print("Example tokens:", train_tokens[0][:20])

Example tokens: ['लोगों', 'को', 'बिलों', 'संबंधी', 'सुविधा', 'देना', 'ही', 'उनका', 'काम']


In [23]:
from collections import Counter

def get_ngrams(tokens, n=1):
    ngrams = []
    for sentence in tokens:
        for i in range(len(sentence)-n+1):
            ngrams.append(tuple(sentence[i:i+n]))
    return ngrams

# Count unigrams and bigrams
unigrams = Counter(get_ngrams(train_tokens, 1))
bigrams  = Counter(get_ngrams(train_tokens, 2))

print("Top 10 unigrams:", unigrams.most_common(10))
print("Top 10 bigrams:", bigrams.most_common(10))

Top 10 unigrams: [(('के',), 2363), (('में',), 1821), (('की',), 1478), ((',',), 1177), (('से',), 1104), (('को',), 1067), (('ने',), 760), (('है',), 752), (('है।',), 751), (('का',), 738)]
Top 10 bigrams: [(('के', 'लिए'), 372), (('है', 'कि'), 210), (('है', '.'), 161), (('है', ','), 127), (('के', 'साथ'), 120), (('के', 'बाद'), 120), (('कहा', 'कि'), 113), (('करने', 'के'), 80), (('ने', 'कहा'), 76), (('हैं', ','), 73)]


In [24]:
def compute_pmi(unigrams, bigrams, total_tokens):
    pmi_scores = {}
    for (w1, w2), bigram_count in bigrams.items():
        p_w1 = unigrams[(w1,)] / total_tokens
        p_w2 = unigrams[(w2,)] / total_tokens
        p_w1w2 = bigram_count / total_tokens
        pmi = math.log2(p_w1w2 / (p_w1 * p_w2)) if p_w1w2 > 0 else 0
        pmi_scores[(w1, w2)] = pmi
    return pmi_scores

total_tokens = sum(unigrams.values())
pmi_scores = compute_pmi(unigrams, bigrams, total_tokens)

# Show top 10 PMI bigrams
sorted_pmi = sorted(pmi_scores.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top PMI bigrams:", sorted_pmi)

Top PMI bigrams: [(('इनेलो', '1987'), 15.889456430504008), (('चैम्पियंस', 'ट्राफी'), 15.889456430504008), (('पैट्रीकियो', 'रोसेंडे'), 15.889456430504008), (('धनंजय', 'देवांगन'), 15.889456430504008), (('PACIFIC', 'ROYAL'), 15.889456430504008), (('ROYAL', 'AIRLINES'), 15.889456430504008), (('दोहरे', 'रिटर्न'), 15.889456430504008), (('अबतक', 'खामोश'), 15.889456430504008), (('रियान', 'पराग'), 15.889456430504008), (('गुलाबी', 'मीनाकारी'), 15.889456430504008)]


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Make sure sentences exist
print("Number of training sentences:", len(train_sentences))
print("Example sentence:", train_sentences[0] if len(train_sentences) > 0 else "EMPTY")

# Filter out empty strings
train_sentences = [s for s in train_sentences if s.strip()]
valid_sentences = [s for s in valid_sentences if s.strip()]
test_sentences  = [s for s in test_sentences  if s.strip()]

# Train TF-IDF on train data
vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")  # <-- ensures Hindi tokens also counted
X_train = vectorizer.fit_transform(train_sentences)

# Transform validation and test using same vocab & IDF
X_valid = vectorizer.transform(valid_sentences)
X_test  = vectorizer.transform(test_sentences)

print("Shape of Train:", X_train.shape)
print("Shape of Valid:", X_valid.shape)
print("Shape of Test :", X_test.shape)

Number of training sentences: 1433
Example sentence: ल   ो   ग   ो   ं   क   ो   ब   ि   ल   ो   ं   स   ं   ब   ं   ध   ी   स   ु   व   ि   ध   ा   द   े   न   ा   ह   ी   उ न क   ा   क   ा   म
Shape of Train: (1433, 91)
Shape of Valid: (346, 91)
Shape of Test : (347, 91)


In [26]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Example: find nearest neighbor in VALIDATION set
# X_valid: TF-IDF matrix of shape (num_sentences, num_features)

# Compute cosine similarity between all pairs
similarity_matrix = cosine_similarity(X_valid)

# To avoid a sentence being its own neighbor, set diagonal to -1
np.fill_diagonal(similarity_matrix, -1)

# Find index of nearest neighbor for each sentence
nearest_neighbors = similarity_matrix.argmax(axis=1)

# Show example
for i in range(5):
    print(f"Sentence {i} nearest neighbor index: {nearest_neighbors[i]}")

Sentence 0 nearest neighbor index: 220
Sentence 1 nearest neighbor index: 10
Sentence 2 nearest neighbor index: 165
Sentence 3 nearest neighbor index: 311
Sentence 4 nearest neighbor index: 147


In [27]:
# Same procedure for TEST set
similarity_matrix_test = cosine_similarity(X_test)
np.fill_diagonal(similarity_matrix_test, -1)
nearest_neighbors_test = similarity_matrix_test.argmax(axis=1)

for i in range(5):
    print(f"Test sentence {i} nearest neighbor index: {nearest_neighbors_test[i]}")

Test sentence 0 nearest neighbor index: 345
Test sentence 1 nearest neighbor index: 36
Test sentence 2 nearest neighbor index: 84
Test sentence 3 nearest neighbor index: 22
Test sentence 4 nearest neighbor index: 105


In [29]:
import pandas as pd
import numpy as np

# Use .shape[0] instead of len()
df_valid_nn = pd.DataFrame({
    "sentence_index": np.arange(X_valid.shape[0]),
    "nearest_neighbor_index": nearest_neighbors
})
df_valid_nn.to_csv("nearest_neighbors_valid.csv", index=False, encoding="utf-8")

df_test_nn = pd.DataFrame({
    "sentence_index": np.arange(X_test.shape[0]),
    "nearest_neighbor_index": nearest_neighbors_test
})
df_test_nn.to_csv("nearest_neighbors_test.csv", index=False, encoding="utf-8")

print("✅ Nearest neighbor indices saved for validation and test sets")

✅ Nearest neighbor indices saved for validation and test sets


In [31]:
from google.colab import files

# Example: download nearest neighbor CSV
files.download("nearest_neighbors_valid.csv")
files.download("nearest_neighbors_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>